# 03 — Test predictive value

We test whether three features derived from past SPY prices help classify a future price movement. Seven fixed classifiers are compared with a simple training-majority baseline using chronological training and validation periods.

## 1. Data and setup

Load daily SPY closes from the [included historical panel](../../data/raw/tpq_eod_2014_2023.csv) through **2021-12-28**. This cutoff also bounds the future closes available to construct labels.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/quant_journey').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start within the teaching checkout or runner snapshot.')
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data/raw/tpq_eod_2014_2023.csv'
OUT = ROOT / 'outputs/track_e_multifamily'
OUT.mkdir(parents=True, exist_ok=True)
from src.quant_journey.execution.prediction_handoff import CUTOFF

In [ ]:
import hashlib
import json
from src.quant_journey.execution.prediction_handoff import MODELS, VALIDATION_START, RESERVED_START, FEATURE_COLUMNS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from src.quant_journey.data.tpq import load_spy_prices_through
from src.quant_journey.execution.dataset import make_execution_dataset
from src.quant_journey.validation.label_aware import split_train_validation_by_dates

prices = load_spy_prices_through(DATA, CUTOFF)

## 2. Define the prediction question

At the end of session *t*, use only information then available to classify whether SPY rises from close $t+1$ to close $t+2$:

$$
y_t = \mathbf{1}\left[\frac{P_{t+2}}{P_{t+1}} - 1 > 0\right]
$$

Define $y_t=1$ as a positive movement and $y_t=0$ otherwise. The target is observed from close *t+1* to close *t+2*. This matches Notebook 04’s hypothetical timing: enter at the next close, then value the position at the following close. It is a close-only price-change proxy, not a claim about an achievable fill.

## 3. Construct the three predictors

All three inputs use SPY closes available by the end of session $t$. Let $P_t$ denote the close and define the one-session percentage change as $r_t = \frac{P_t}{P_{t-1}} - 1$:

- `return_1d`: $r_t$, the latest one-session percentage change.
- `momentum_5d`: $\frac{P_t}{P_{t-5}} - 1$, the five-session close-to-close percentage change.
- `volatility_5d`: the sample standard deviation (Pandas `ddof=1`) of $r_{t-4},\ldots,r_t$, the latest five one-session percentage changes.

Notebook 02’s SMA, RSI, MACD, PCA and clusters are exploratory descriptions; none is an input to this fixed comparison.

In [ ]:
data = make_execution_dataset(prices)

## 4. Separate training and validation

Use feature dates before **2020-01-02** for training and dates from **2020-01-02** to before **2021-12-29** for validation. Keep a training label only when its `label_available_at` date is strictly before validation starts, removing training labels that mature into validation. Validation labels must mature before the reserved boundary. Training contains 1,503 rows (**2014-01-09–2019-12-27**); validation contains 500 rows (**2020-01-02–2021-12-23**). Any standardizer is inside a model pipeline and is fitted on training rows only. Validation has previously been examined; reserved observations were excluded from model development and reported evaluation.

In [ ]:
train, validation = split_train_validation_by_dates(
    data, VALIDATION_START, RESERVED_START
)
features = list(FEATURE_COLUMNS)
target = 'execution_target_up'
X_train = train[features]
y_train = train[target]
X_val = validation[features]
y_val = validation[target]
print(
    'train/validation:', X_train.shape, X_val.shape,
    'date ranges:', train.index.min().date(), train.index.max().date(),
    validation.index.min().date(), validation.index.max().date(),
)
assert (len(train), len(validation)) == (1503, 500)
assert np.isfinite(X_train).all().all() and np.isfinite(X_val).all().all()

**Interpretation.** Feature dates determine the initial partitions. The purge then retains training rows with `label_available_at < 2020-01-02` and validation rows with `label_available_at < 2021-12-29`. A label that matures on the boundary is excluded. Validation is prior development evidence, not a fresh holdout.

## 5. Define the classifiers and benchmark

Compare GaussianNB, logistic regression, a decision tree, a random forest, LinearSVC, a hard-voting ensemble and a multilayer perceptron (MLP). The pipelines fit their standardizers on training rows. The hard vote combines GaussianNB, logistic regression, the forest and LinearSVC.

The benchmark predicts the most frequent training label on every validation date. All estimator settings are fixed before this comparison.

In [ ]:
# Fixed canonical estimator and preprocessing settings.
nb = GaussianNB()
logit = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=500, solver='lbfgs', random_state=42),
)
tree = DecisionTreeClassifier(random_state=42)
forest = RandomForestClassifier(
    n_estimators=30, max_depth=9, random_state=42, n_jobs=1
)
svc = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0, dual=False, max_iter=10000, class_weight='balanced',
        random_state=42,
    ),
)

### Hard vote and neural network

In [ ]:
voting = VotingClassifier(
    estimators=[('nb', nb), ('logit', logit), ('rf', forest), ('svc', svc)],
    voting='hard', n_jobs=1,
)
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(32, 24, 16), activation='relu', solver='adam',
        learning_rate_init=.001, max_iter=500, random_state=42,
    ),
)

### Fitting order and estimator settings

In [ ]:
specifications = [
    (nb, 'sklearn defaults'),
    (logit, 'C=1; lbfgs; max_iter=500; scaled'),
    (tree, 'sklearn defaults; seed=42'),
    (forest, '30 trees; max_depth=9; seed=42'),
    (svc, 'C=1; dual=False; max_iter=10000; balanced; scaled'),
    (voting, 'hard vote: NB, logistic, RF, LinearSVC'),
    (mlp, '32-24-16; relu; adam; lr=.001; max_iter=500; scaled; seed=42'),
]
models = dict(zip(MODELS[:-1], [model for model, _ in specifications], strict=True))
settings = pd.DataFrame({
    'family': list(models),
    'fixed settings': [description for _, description in specifications],
})
display(settings)

## 6. Fit, predict and compare

Fit each family once on the shared training rows and predict the same validation dates. **Accuracy** is the fraction of validation labels classified correctly. **Balanced accuracy** averages recall across the two classes, giving each class equal weight. Each confusion matrix shows actual classes by rows and predicted classes by columns; all seven classifiers and the training-majority baseline are shown.

In [ ]:
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    # Each fitted classifier predicts on the same validation rows.
    predictions[name] = pd.Series(
        model.predict(X_val).astype(int), index=X_val.index, name=name
    )
predictions[MODELS[-1]] = pd.Series(
    int(y_train.mode().iloc[0]), index=X_val.index
)

### Score the validation predictions

In [ ]:
metrics = []
for name, pred in predictions.items():
    metrics.append({
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'balanced_accuracy': balanced_accuracy_score(y_val, pred),
        'predicted_up': int(pred.sum()),
    })
metrics = pd.DataFrame(metrics).set_index('model')
display(metrics.assign(accuracy=metrics.accuracy.map('{:.2%}'.format),
                       balanced_accuracy=metrics.balanced_accuracy.map('{:.2%}'.format)))
metrics.to_csv(OUT / 'validation_metrics.csv', index_label='model')

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(metrics.balanced_accuracy, metrics.index)
ax.axvline(.5, color='gray', linestyle='--', label='Majority baseline (50%)')
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.set(xlabel='Validation balanced accuracy', xlim=(.47, .53))
ax.legend()
fig.tight_layout()
plt.show()

**Validation results.** The training-majority benchmark predicts UP on all 500 validation dates and is correct on 290 (58.0%). Logistic regression makes the same predictions. All six other classifiers have lower accuracy. With a 500-epoch cap, the MLP predicts UP 285 times, with accuracy 0.502 and balanced accuracy 0.490558.

**Numerical reproducibility.** Apple Accelerate emitted matrix-operation warnings of undetermined cause. An independent OpenBLAS comparison reproduced predictions for an earlier configuration. The reported cap-500 validation predictions were checked on the project backend.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (name, pred) in zip(axes.flat, predictions.items()):
    matrix = confusion_matrix(y_val, pred, labels=[0, 1])
    ax.imshow(matrix, cmap='Blues')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('predicted')
    ax.set_ylabel('actual')
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha='center', va='center')
plt.tight_layout()

**Interpretation.** The panels keep each fitted family beside the simple reference. The all-UP logistic and baseline panels make their identical class predictions visible, including the 210 validation non-UP labels they miss. In the revised MLP panel, the actual non-UP row contains 88 predicted non-UP and 122 predicted UP. The actual UP row contains 127 predicted non-UP and 163 predicted UP.

## 7. Save predictions for portfolio accounting

We save each model’s prediction by feature date, alongside the observed label. Notebook 04 loads these predictions to construct hypothetical positions. Separate files record classification metrics and the experiment configuration. The manifest copies the expected snapshot digest into `data_sha256`. The producing workflow reads only the bounded teaching period.

In [ ]:
# Persist predictions and provenance.
pred_path = OUT / 'validation_predictions.csv'
pd.DataFrame(predictions).assign(actual=y_val).to_csv(
    pred_path,
    index_label='feature_date',
)
manifest = {
    'validation_predictions_csv_sha256': hashlib.sha256(pred_path.read_bytes()).hexdigest(),
    'data_sha256': '771c40d3f8d19d1cf524764296bfa2943f8ca9c63514b99bb8d95c9b84d1e407',
    'cutoff': CUTOFF,
    'features': features,
    'target': 'close[t+2]/close[t+1]-1 > 0',
    'train_rows': len(train),
    'validation_rows': len(validation),
    'models': settings.to_dict(orient='records'),
    'reserved_test_access': False,
    'validation_is_fresh_holdout': False,
}
_ = (OUT / 'provenance.json').write_text(
    json.dumps(manifest, indent=2)
)

The prediction CSV records one row per feature date, with model predictions and the observed label. The manifest stores its SHA-256 digest so Notebook 04 can verify the exact input before accounting.

## Conclusion

The small differences in balanced accuracy on these previously examined validation dates do not establish predictive skill. Notebook 04 uses the dated predictions to examine how execution timing and trading costs affect account value.